<a href="https://colab.research.google.com/github/qossiim/computer_vision_final/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print('Menu Detector!')

Menu Detector!


In [ ]:
# -----------------------------------
# Import Libraries
# -----------------------------------
from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
# ===========================
# Define Dataset Path
# ===========================

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print('Dataset_path:', DATASET_PATH)

CUSTOM_CLASS_MAPPING = {
    "hamburger": "hamburger",
    "hot_dog": "hot_dog",
    "chocolate_cake": "dessert",   # label grouping | class consolidation
    "cheesecake": "dessert",       # label grouping | class consolidation
    "kebab": "kebab",
    "pilaf": "pilaf"
}

CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']

CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}

NUM_CLASSES = len(CLASSES)

print(CLASS_TO_IDX)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])





In [ ]:
# ============================
# Custom Dataset Class
# ============================

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        print('images_length', len(self.images))
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        print('image_path', img_path)

        label = self.labels[idx]
        print('label', label)

        try:
            image = Image.open(img_path).convert('RGB')
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self.images))

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
# ============================
# Gather and Split Data
# ============================

all_images = []

for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)
    print('class_path:', class_path)

    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue

    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(class_path, img)
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)

split = int(0.8 * len(all_images))

train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[0]

In [ ]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# pretrained model
model = mobilenet_v2(weights='IMAGENET1K_V1')
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()  # Loss Function | '70%' burger, '30%' pilaf

optimizer = optim.Adam(model.parameters(), lr=0.001)  # weight

torch.backends.cudnn.benchmark = True  # Benchmark Setting | Trick | 10%-20%

In [ ]:
import time

images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

model = model.to(device)

print("Before forward")

start = time.time()

outputs = model(images)

torch.cuda.synchronize()

print("After forward")
print("Forward time:", time.time() - start)

In [ ]:
''' 8 - TRAINING LOOP & SAVE EFFICIENT AI MODEL '''

NUM_EPOCHS = 2
best_accuracy = 0.0

# Move model to GPU/CPU
model = model.to(device)

for epoch in range(NUM_EPOCHS):

    # -------------------- TRAIN --------------------
    model.train()
    running_loss = 0.0

    for batch_idx, (images, labels) in enumerate(train_loader):

        # Move data to GPU/CPU
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()          # Zero the gradients

        outputs = model(images)        # Forward Pass

        loss = criterion(outputs, labels)  # Calculate Loss

        loss.backward()                # Backpropagation

        optimizer.step()               # Update Weights

        running_loss += loss.item()

        # Show training progress every 10 batches
        if (batch_idx + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1}/{NUM_EPOCHS} | "
                f"Batch {batch_idx+1}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    # -------------------- VALIDATION --------------------
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    # Calculate Validation Accuracy
    val_acc = 100 * correct / total

    # Print Epoch Result
    print(
        f"\nEpoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Loss: {running_loss / len(train_loader):.4f}, "
        f"Val Accuracy: {val_acc:.2f}%\n"
    )

    # Save Best Model
    if val_acc > best_accuracy:

        best_accuracy = val_acc

        torch.save(model.state_dict(), "/content/menu_detector.pth")

        print(" Saved new best model!\n")

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2
from PIL import Image
from google.colab import files
import io
import matplotlib.pyplot as plt


''' 1 - DEFINE SEMANTIC CLASSES (LABELS) '''


CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']  # Must match training order
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}


''' 2 - TRANSFORM FOR UPLOADED IMAGES '''


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


''' 3 - LOADING MODEL '''


model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model.load_state_dict(torch.load('/content/menu_detector.pth', map_location='cpu'))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

model = model.to(device)
model.eval()


''' 4 - UPLOAD TO PREDICT '''


print("Upload one or more images of your food:")
uploaded = files.upload()

for image_name in uploaded.keys():
    image = Image.open(io.BytesIO(uploaded[image_name])).convert('RGB')

    # Display image
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f'Uploaded: {image_name}')
    plt.show()

    # Predict
    image_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(image_tensor)
        probs = torch.softmax(output, dim=1)[0]
        topk = torch.topk(probs, 4)

    print("Prediction:")
    for i in range(topk.indices.size(0)):
        label = CLASSES[topk.indices[i]]
        confidence = topk.values[i].item() * 100
        print(f"\t✅ {label}: {confidence:.2f}%")